In [2]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import datasets, layers, models, losses
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
import tempfile
from matplotlib import pyplot as plt 
import numpy as np 
from keras.layers import Conv1D,Conv2D,Reshape,GlobalAveragePooling2D,Multiply,Lambda,Input, Activation,GlobalMaxPooling2D,Dense,MaxPooling2D,Flatten
from tensorflow.keras.applications.vgg19 import VGG19
from keras.utils.np_utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, LearningRateScheduler
import gc

In [13]:
# Set the random seed
import os
import ssl
os.environ['PYTHONHASHSEED'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = ''
np.random.seed(37)
tf.random.set_seed(89)

ssl._create_default_https_context = ssl._create_unverified_context
#------------------------------------------------------------------------------
#  Loading CIFAR10 data
#------------------------------------------------------------------------------

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Convert class vectors to binary class matrices using one hot encoding
y_train_ohe = to_categorical(y_train, num_classes = 10)
y_test_ohe = to_categorical(y_test, num_classes = 10)

# Data normalization
x_train=tf.image.resize(x_train,(48,48))
x_test=tf.image.resize(x_test,(48,48))
x_train = x_train.numpy()
x_test = x_test.numpy()
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train  /= 255
x_test /= 255

#dataset splitting train-val- test
x_val = x_train[40000:]
y_val = y_train_ohe[40000:]
print(x_val.shape)  
print(y_val.shape)

x_train = x_train[:40000]
y_train_ohe = y_train_ohe[:40000]
print(x_train.shape)
print(y_train_ohe.shape)  

(10000, 48, 48, 3)
(10000, 10)
(40000, 48, 48, 3)
(40000, 10)


In [9]:
vgg16_model = VGG16(weights='imagenet',
                    include_top=False, 
                    classes=10,
                    input_shape=(48,48,3)# input: images with 3 channels -> (224,224,3) tensors.
                   )

#Define the sequential model and add th VGG's layers to it
from tensorflow.keras.models import Sequential
model = Sequential()
for layer in vgg16_model.layers:
    model.add(layer)

from tensorflow.keras.layers import Dense, Flatten, Dropout
model.add(Flatten())
#model.add(Dense(512, activation='relu'))
#model.add(Dropout(0.2))
model.add(Dense(512, activation='relu'))
model.add(Dense(512, activation='relu'))
#model.add(Dropout(0.2))
model.add(Dense(10, activation='softmax'))

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 block1_conv1 (Conv2D)       (None, 48, 48, 64)        1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 48, 48, 64)        36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 24, 24, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 24, 24, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 24, 24, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 12, 12, 128)       0         
                                                                 
 block3_conv1 (Conv2D)       (None, 12, 12, 256)       2

In [10]:
#------------------------------------------------------------------------------
# TRAINING THE CNN ON THE TRAIN/VALIDATION DATA
#------------------------------------------------------------------------------
from tensorflow.keras import optimizers
# initiate SGD optimizer
sgd = optimizers.SGD(lr=0.001, momentum=0.9)

# For a multi-class classification problem
model.compile(loss='categorical_crossentropy',optimizer= sgd,metrics=['accuracy'])


def lr_scheduler(epoch):
    return 0.001 * (0.5 ** (epoch // 20))
reduce_lr = LearningRateScheduler(lr_scheduler)

#mc = ModelCheckpoint('./weights.h5', monitor='val_accuracy', save_best_only=True, mode='max')
Monitor=[
    #tf.keras.callbacks.EarlyStopping(monitor='val_accuracy',mode='max',patience=10),
    tf.keras.callbacks.ModelCheckpoint('model.h5',monitor='val_accuracy',mode='max',save_best_only=True)
]

# initialize the number of epochs and batch size
EPOCHS = 100
BS = 8

# construct the training image generator for data augmentation
aug = ImageDataGenerator(
    rotation_range=20, 
    zoom_range=0.15, 
    width_shift_range=0.2, 
    height_shift_range=0.2, 
    shear_range=0.15,
    horizontal_flip=True, 
    fill_mode="nearest")
 
# train the model
#history = model.fit_generator(aug.flow(x_train,y_train_ohe, batch_size=BS,seed=42),validation_data=(x_val,y_val),
    #steps_per_epoch=len(x_train) // BS,epochs=EPOCHS,callbacks=[reduce_lr,Monitor])

C:\Users\Student\.conda\envs\abc\lib\site-packages\keras\optimizer_v2\gradient_descent.py:102: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


In [11]:
#model.load_weights('model_cifar10.ckpt')
model.load_weights('model_cifar10.ckpt')

In [12]:
#mdl=tf.keras.models.load_model("model.h5")
model_loss, baseline_model_accuracy = model.evaluate(
     x_test,y_test_ohe, verbose=1)

313/313 [==============================] - 2s 4ms/step - loss: 0.3463 - accuracy: 0.9373


Baseline test loss:  accuracy: 0.9111999869346619
[0.9111999869346619]
[21, 48, 48, 48, 60, 70, 70, 100, 65, 65, 44, 44, 56]
Total number of parameters before pruning: 15245130
Total number of parameters after pruning: 681505
parameter reduction: 95.52968718535034


# Trancation

In [17]:
import numpy as np
import os
from codecs import decode
import struct


#os.environ["CUDA_VISIBLE_DEVICES"] = "-1" #Disables GPU acceleration

global count_total_bits_truncated
count_total_bits_truncated = 0

def read_args():
    f = open('args.txt')
    args = f.read().split('\n')
    f.close()
    return args


def truncate_weights(model, args):
    def bin_truncation(binary_string, n_bits):
        if n_bits == 0: #Error Condition avoidance
            return binary_string

        #fill truncation values
        result = binary_string[:-n_bits] + '1'
        while (len(result) < 32):
            result = result + '0'

        return (result)

    def bin_to_float(b):
        """ Convert binary string to a float. """
        b = b + '00000000000000000000000000000000'  # convert back to 64 bit
        bf = int_to_bytes(int(b, 2), 8)  # 8 bytes needed for IEEE 754 binary64.
        return struct.unpack('>d', bf)[0]

    def int_to_bytes(n, length):  # Helper function
        """ Int/long to byte string.

            Python 3.2+ has a built-in int.to_bytes() method that could be used
            instead, but the following works in earlier versions including 2.x.
        """
        return decode('%%0%dx' % (length << 1) % n, 'hex')[-length:]

    def float_to_bin(value):  # For testing.
        """ Convert float to 32-bit binary string. """
        [d] = struct.unpack(">Q", struct.pack(">d", value))
        return '{:064b}'.format(d)[:32]

    def truncation_process(weight, n_bits):
        a = float_to_bin(weight)
        b = bin_truncation(a, n_bits)
        return bin_to_float(b)

    def recursive_truncate(node, n_bits):
        #if list, dig deeper in recursion
        if 'list' in str(type(node)):
            for count, i in enumerate(node):
                node[count] = recursive_truncate(node[count], n_bits)
            return node
        #if ndarray, dig deeper in recursion
        elif 'ndarray' in str(type(node)):
            for count, i in enumerate(node):
                node[count] = recursive_truncate(node[count], n_bits)
            return node
        #finally, if single weight. truncate and return up recusion
        else:
            global count_total_bits_truncated
            count_total_bits_truncated = count_total_bits_truncated + n_bits
            return truncation_process(node, n_bits)

    def truncation_handler(layer, n_bits):
        def convolution_handler(layer):
            weights = layer.get_weights()
            print('CNN truncation: n_bits = ', n_bits)
            new_weights = recursive_truncate(weights, n_bits)
            return new_weights

        def dense_handler(layer):
            weights = layer.get_weights()
            print('ANN truncation: n_bits = ', n_bits)
            new_weights = recursive_truncate(weights, n_bits)
            return new_weights


        s = str(type(layer))

        if 'Conv2D' in s:
            print('Convolution')
            return convolution_handler(layer)

        elif 'Dense' in s:
            print('DENSE')
            return dense_handler(layer)

        else:
            print(type(layer))
            print(layer.get_weights())
            print(s)
            print('LAYER UNEXPECTED')
            exit()

    def check_skip_layers(layer):
        s = str(type(layer))
        if 'convolutional.UpSampling2D' in s:
            print('action: PASS - up sampling')
            return True

        elif 'InputLayer' in s:
            print('action: PASS - InputLayer')
            return True
        elif 'ZeroPadding2D' in s:
            print('action: PASS - ZeroPadding2D')
            return True

        elif 'MaxPooling2D' in s:
            print('actoin: PASS - MaxPooling2D')
            return True

        elif 'Dropout' in s:
            print('action: PASS - dropout')
            return True

        elif 'Flatten' in s:
            print('action: PASS - Flatten')
            return True
        elif 'UpSampling' in s:
            print('action: PASS - UpSampiling')
            return True
        elif 'BatchNormalization' in s:
            print('BATCH NORMALIZATION')
            return True

        else:
            return False


    #begin truncation
    layerTruncationIndex = 0

    print('Model has Layers = ', len(model.layers))
    for count, layer in enumerate(model.layers):
        if check_skip_layers(layer):
            pass
        else:
            #get n_bits via a global counter
            n_bits = int(args[layerTruncationIndex])
            new_layer = truncation_handler(layer, n_bits)
            model.layers[count].set_weights(new_layer)
            layerTruncationIndex = layerTruncationIndex + 1

        print('*************************************')


    return model


def test_model(model, x_train, y_train, x_test, y_test, args, file_name='Model'):

    print('Begin Testing....')
    total_correct = 0
    total_possible = y_train.shape[0] + y_test.shape[0]

    predictions = np.argmax(model.predict(x_train), axis=-1)
    for i in range(predictions.shape[0]):
        if (predictions[i] == np.argmax(y_train[i])):
            total_correct = total_correct + 1

    predictions = np.argmax(model.predict(x_test), axis=-1)

    for i in range(predictions.shape[0]):
        if (predictions[i] == np.argmax(y_test[i])):
            total_correct = total_correct + 1

    print('Total Correct: ', total_correct)
    print('Total Possibl: ', total_possible)

    print('Accuracy = ', (total_correct/total_possible))
    print(count_total_bits_truncated)

    #Save all of the important information into csv file
    for i in args:
        file_name = file_name + '_' + str(i)
    file_name = file_name + '.csv'

    f = open(file_name, "a")
    for i in args:
        f.write(str(i) + ',')
    f.write(str(total_correct) + ',' + str(total_possible) + ',' + str(count_total_bits_truncated))
    f.close()


In [14]:
model.save('model.h5') 
#import tensorflow
#import Models
#import keras
#import numpy as np
import os
#import TruncationEmulator
#from tensorflow.keras.optimizers import RMSprop

def main():
    #model = Models.load_model_resNet50(input_shape=(32, 32, 3))
    #model = Models.load_model_alexnet(input_shape=(32, 32, 3))
    #x_train, y_train, x_test, y_test = Models.load_data()

    #model.compile(loss='categorical_crossentropy',
                 # optimizer=tensorflow.keras.optimizers.RMSprop(learning_rate=2e-5),
                 # metrics=['accuracy'])

    model.load_weights('model.h5') 
    model.summary()
    model.evaluate(x_test, y_test_ohe, verbose=1) 
    variables = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15, 16,17,18,19,20,21,22,23,24,25,26,27,28,29,30, 31]
    #variables=[0,16,31]
    for i1 in variables:
        args = [i1, i1, i1, i1, i1, i1, i1,i1, i1, i1, i1, i1, i1, i1,i1, 0] #spesific to AlexNet, makes it so the final output layer is never truncated

        truncate_model = truncate_weights(model, args)

        test_model(truncate_model, x_test, y_test_ohe, x_test, y_test_ohe, args, 'model')



if __name__ == "__main__":
    main()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 block1_conv1 (Conv2D)       (None, 48, 48, 64)        1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 48, 48, 64)        36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 24, 24, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 24, 24, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 24, 24, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 12, 12, 128)       0         
                                                                 
 block3_conv1 (Conv2D)       (None, 12, 12, 256)       2